# Clean and Analyze Employee Exit Surveys

In this project, we work with exit surveys from employees of the Department of Education, Training and Employment (DETE) and the Technical and Further Education (TAFE) institute in Queensland, Australia. Both surveys used the same template, but TAFE customized some of the answers, so the data needs substantial cleaning before the two can be combined.

We play the role of a data analyst whose stakeholders want to know:

- Are employees who only worked for the institutes for a short period of time resigning due to some kind of dissatisfaction? What about employees who have been there longer?
- Are younger employees resigning due to some kind of dissatisfaction? What about older employees?

In the guided steps we focus on the first question.

## Reading and Exploring the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

dete_survey = pd.read_csv('dete_survey.csv')
tafe_survey = pd.read_csv('tafe_survey.csv')

In [ ]:
dete_survey.info()

In [ ]:
dete_survey.head()

In [ ]:
tafe_survey.info()

In [ ]:
tafe_survey.head()

In [ ]:
dete_survey.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
tafe_survey.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
dete_survey['SeparationType'].value_counts()

In [ ]:
tafe_survey['Reason for ceasing employment'].value_counts()

A few observations:

- `dete_survey` contains `'Not Stated'` values that indicate missing data, but they aren't represented as `NaN`.
- Both dataframes contain many columns we don't need to answer our questions.
- Both dataframes contain many of the same columns, but with different names.
- Several columns/answers indicate that an employee resigned because they were dissatisfied.

## Identify Missing Values and Drop Unnecessary Columns

In [ ]:
dete_survey = pd.read_csv('dete_survey.csv', na_values='Not Stated')

dete_survey_updated = dete_survey.drop(dete_survey.columns[28:49], axis=1)
tafe_survey_updated = tafe_survey.drop(tafe_survey.columns[17:66], axis=1)

print(dete_survey_updated.shape)
print(tafe_survey_updated.shape)

We re-read `dete_survey` so that `'Not Stated'` is treated as `NaN`. Then we dropped columns 28–48 from `dete_survey` and columns 17–65 from `tafe_survey`: they hold detailed opinion questions about the workplace that we won't use to answer our questions, and removing them makes the dataframes much easier to work with.

## Clean Column Names

In [ ]:
dete_survey_updated.columns = dete_survey_updated.columns.str.lower().str.strip().str.replace(' ', '_')
dete_survey_updated.columns

In [ ]:
mapping = {
    'Record ID': 'id',
    'CESSATION YEAR': 'cease_date',
    'Reason for ceasing employment': 'separationtype',
    'Gender. What is your Gender?': 'gender',
    'CurrentAge. Current Age': 'age',
    'Employment Type. Employment Type': 'employment_status',
    'Classification. Classification': 'position',
    'LengthofServiceOverall. Overall Length of Service at Institute (in years)': 'institute_service',
    'LengthofServiceCurrent. Length of Service at current workplace (in years)': 'role_service'
}
tafe_survey_updated = tafe_survey_updated.rename(mapping, axis=1)
tafe_survey_updated.columns

In [ ]:
dete_survey_updated.head()

In [ ]:
tafe_survey_updated.head()

Since we'll eventually combine the two dataframes, we standardized the column names: the DETE columns are now lowercase, stripped, and use underscores instead of spaces, and the TAFE columns we need were renamed to match them (`id`, `cease_date`, `separationtype`, `gender`, `age`, etc.).

## Filter the Data

In [ ]:
dete_survey_updated['separationtype'].value_counts()

In [ ]:
tafe_survey_updated['separationtype'].value_counts()

In [ ]:
dete_resignations = dete_survey_updated[dete_survey_updated['separationtype'].str.contains('Resignation', na=False)].copy()
tafe_resignations = tafe_survey_updated[tafe_survey_updated['separationtype'] == 'Resignation'].copy()

print(dete_resignations.shape)
print(tafe_resignations.shape)

We only want to analyze employees who resigned. DETE has three resignation categories (`Resignation-Other reasons`, `Resignation-Other employer`, `Resignation-Move overseas/interstate`), so we selected every separation type containing the string `'Resignation'`. TAFE has a single `'Resignation'` category.

## Verify the Data

In [ ]:
dete_resignations['cease_date'].value_counts()

In [ ]:
dete_resignations['cease_date'] = dete_resignations['cease_date'].str.split('/').str[-1].astype('float')
dete_resignations['cease_date'].value_counts().sort_index()

In [ ]:
dete_resignations['dete_start_date'].value_counts().sort_index()

In [ ]:
tafe_resignations['cease_date'].value_counts().sort_index()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
dete_resignations.boxplot(column='cease_date', ax=axes[0])
dete_resignations.boxplot(column='dete_start_date', ax=axes[1])
tafe_resignations.boxplot(column='cease_date', ax=axes[2])
axes[0].set_title('DETE cease date')
axes[1].set_title('DETE start date')
axes[2].set_title('TAFE cease date')
plt.show()

Some DETE cease dates were stored as `month/year`, so we kept only the year and converted the column to float. The years look reasonable: no cease date is later than the current date, and no start date is before 1940. The two surveys don't cover exactly the same years (DETE resignations are mostly 2012–2014, TAFE 2009–2013), but since we aren't analyzing results by year, we'll keep all of them.

## Create a New Column: Years of Service

In [ ]:
dete_resignations['institute_service'] = dete_resignations['cease_date'] - dete_resignations['dete_start_date']
dete_resignations['institute_service'].value_counts().sort_index().head(10)

TAFE already has an `institute_service` column (length of service in years). To analyze both surveys together, we created the same column for DETE by subtracting the start year from the cease year.

## Identify Dissatisfied Employees

In [ ]:
tafe_resignations['Contributing Factors. Dissatisfaction'].value_counts(dropna=False)

In [ ]:
tafe_resignations['Contributing Factors. Job Dissatisfaction'].value_counts(dropna=False)

In [ ]:
def update_vals(val):
    if pd.isnull(val):
        return np.nan
    elif val == '-':
        return False
    else:
        return True

tafe_resignations['dissatisfied'] = (
    tafe_resignations[['Contributing Factors. Dissatisfaction', 'Contributing Factors. Job Dissatisfaction']]
    .apply(lambda col: col.map(update_vals))
    .any(axis=1, skipna=False)
)
tafe_resignations_up = tafe_resignations.copy()
tafe_resignations_up['dissatisfied'].value_counts(dropna=False)

In [ ]:
dete_factors = [
    'job_dissatisfaction', 'dissatisfaction_with_the_department', 'physical_work_environment',
    'lack_of_recognition', 'lack_of_job_security', 'work_location',
    'employment_conditions', 'work_life_balance', 'workload'
]
dete_resignations['dissatisfied'] = dete_resignations[dete_factors].any(axis=1, skipna=False)
dete_resignations_up = dete_resignations.copy()
dete_resignations_up['dissatisfied'].value_counts(dropna=False)

We marked an employee as `dissatisfied` if any of the dissatisfaction-related factors caused them to resign. In TAFE, the two factor columns contain either `'-'` (not a factor) or the factor's name, so we converted them to `False`/`True` first. In DETE, the nine factor columns are already boolean. `DataFrame.any()` then returns `True` if at least one factor is `True`.

## Combine the Data

In [ ]:
dete_resignations_up['institute'] = 'DETE'
tafe_resignations_up['institute'] = 'TAFE'

combined = pd.concat([dete_resignations_up, tafe_resignations_up], ignore_index=True)
combined.notnull().sum().sort_values()

In [ ]:
combined_updated = combined.dropna(thresh=500, axis=1).copy()
combined_updated.columns

We added an `institute` column so we can still tell the surveys apart, stacked the two dataframes, and dropped every column with fewer than 500 non-null values — these are columns that only exist in one of the surveys and that we don't need for the analysis.

## Clean the Service Column

In [ ]:
combined_updated['institute_service'].value_counts(dropna=False)

In [ ]:
combined_updated['institute_service_up'] = (
    combined_updated['institute_service'].astype('str').str.extract(r'(\d+)', expand=False).astype('float')
)
combined_updated['institute_service_up'].value_counts(dropna=False).sort_index()

In [ ]:
def transform_service(val):
    if pd.isnull(val):
        return np.nan
    elif val < 3:
        return 'New'
    elif val < 7:
        return 'Experienced'
    elif val < 11:
        return 'Established'
    else:
        return 'Veteran'

combined_updated['service_cat'] = combined_updated['institute_service_up'].apply(transform_service)
combined_updated['service_cat'].value_counts(dropna=False)

The `institute_service` column mixed exact numbers (DETE) with ranges such as `'1-2'` or `'Less than 1 year'` (TAFE). We extracted the first number from each value, which is a reasonable approximation because the ranges are narrow, and then mapped the years of service into career-stage categories, based on [this article](https://www.businesswire.com/news/home/20171108006002/en/Age-Number-Engage-Employees-Career-Stage):

- **New**: less than 3 years at the company
- **Experienced**: 3–6 years
- **Established**: 7–10 years
- **Veteran**: 11 or more years

## Initial Analysis

In [ ]:
combined_updated['dissatisfied'].value_counts(dropna=False)

In [ ]:
combined_updated['dissatisfied'] = combined_updated['dissatisfied'].fillna(False).astype(bool)

dis_pct = combined_updated.pivot_table(index='service_cat', values='dissatisfied')
dis_pct = dis_pct.reindex(['New', 'Experienced', 'Established', 'Veteran'])
dis_pct

In [ ]:
ax = dis_pct.plot(kind='bar', rot=0, legend=False, ylim=(0, 0.6))
ax.set_title('Share of Resignations Due to Dissatisfaction, by Years of Service')
ax.set_xlabel('Career stage')
ax.set_ylabel('Share dissatisfied')
plt.show()

In [ ]:
combined_updated['service_cat'].value_counts()

## Conclusion

We filled the handful of missing `dissatisfied` values with `False`, the most frequent value, and then computed the share of resigning employees who left due to some kind of dissatisfaction in each career-stage group.

Employees with 7 or more years of service (**Established** and **Veteran**) are the most likely to resign due to some kind of dissatisfaction — roughly half of them do — while less than a third of **New** employees (under 3 years) resign for that reason. **Experienced** employees (3–6 years) sit in between.

So, to answer the stakeholders' first question: short-tenure employees do resign due to dissatisfaction, but noticeably less often than employees who have been at the institutes longer.

This is a preliminary analysis. Next steps would be to check whether the pattern holds when the two institutes are analyzed separately, to handle the remaining missing values more carefully, and to answer the second question by cleaning the `age` column in the same way.